In [1]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.2 MB/s eta 0:00:00


In [5]:
import os
import gc
from pathlib import Path
from tqdm.auto import tqdm
from ultralytics import YOLO
from PIL import Image

# ================================================================= #
# 1. 경로 및 파라미터 설정
# ================================================================= #
model_path = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/formodel/results/runs/bed_seg_2cycle_v1/weights/best.pt"
filelist_dir = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/1. 원본_핵심/2작기/260306-260412_변경후"
savefile_dir = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/0. ROI BOX 원본/이름변경 후_to 0412"

BATCH_SIZE = 32
TOP_TRIM_RATIO = 0.12
BOTTOM_TRIM_RATIO = 0.02
RIGHT_MARGIN = 50

# ================================================================= #
# 2. 실행 함수
# ================================================================= #
def run_final_smart_fix():
    # 1. 파일 목록 확보 및 중복 체크
    valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    raw_paths = [p for p in Path(filelist_dir).rglob('*') if p.suffix.lower() in valid_exts]

    todo_paths = []
    for p in raw_paths:
        rel_path = p.relative_to(filelist_dir)
        out_path = Path(savefile_dir) / rel_path
        if not out_path.exists():
            todo_paths.append(p) # Path 객체 그대로 저장

    total_todo = len(todo_paths)
    print(f"전체: {len(raw_paths)} / 건너뜀: {len(raw_paths)-total_todo} / 신규작업: {total_todo}")

    if total_todo == 0:
        print("작업할 파일이 없습니다.")
        return

    model = YOLO(model_path)
    pbar = tqdm(total=total_todo, desc="ROI 작업 중")

    # 2. 배치 단위 처리
    for i in range(0, total_todo, BATCH_SIZE):
        batch_obj = todo_paths[i : i + BATCH_SIZE]
        batch_strs = [str(p) for p in batch_obj] # 모델 입력용 문자열

        # 추론
        results = model.predict(source=batch_strs, imgsz=1280, conf=0.25, verbose=False)

        # [수정포인트] 원본 경로와 결과를 1:1로 매칭 (zip 사용)
        for original_path, result in zip(batch_obj, results):
            try:
                rel_path = original_path.relative_to(filelist_dir)
                out_path = Path(savefile_dir) / rel_path
                out_path.parent.mkdir(parents=True, exist_ok=True)

                img = Image.open(original_path)
                w, h = img.size

                if len(result.boxes) > 0:
                    box = result.boxes.xyxy[0].cpu().numpy()
                    x1, y1, x2, y2 = box
                    bed_h = y2 - y1

                    crop_y1 = max(0, y1 + (bed_h * TOP_TRIM_RATIO))
                    crop_x2 = min(w, x2 + RIGHT_MARGIN)
                    crop_y2 = min(h, y2 - (bed_h * BOTTOM_TRIM_RATIO))

                    img.crop((0, crop_y1, crop_x2, crop_y2)).save(out_path, quality=90)
                else:
                    img.save(out_path, quality=90)

                img.close()
            except Exception as e:
                print(f"\n파일 오류 ({original_path.name}): {e}")

        # 메모리 비우기
        del results
        gc.collect()
        pbar.update(len(batch_obj))

    pbar.close()
    print("완료되었습니다.")

if __name__ == '__main__':
    run_final_smart_fix()

전체: 2358 / 건너뜀: 404 / 신규작업: 1954


ROI 작업 중:   0%|          | 0/1954 [00:00<?, ?it/s]

완료되었습니다.
